In [45]:
# ROS_DOMAIN_ID 확인
!export ROS_DOMAIN_ID=13 && echo $ROS_DOMAIN_ID

13


In [1]:
!echo $ROS_DOMAIN_ID

13


In [1]:
# commander (PC) : /goal_pose 발행
from nav2_simple_commander.robot_navigator import BasicNavigator
import rclpy

rclpy.init()
nav = BasicNavigator()

In [2]:
# 8
nav.waitUntilNav2Active()

[INFO] [1762516517.424329501] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762516518.426152806] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762516519.429551220] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762516520.431432631] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762516528.451379000] [basic_navigator]: Nav2 is ready for use!


In [3]:
# 9 로봇의 각도를 쿼터니언으로 변환 함수 선언
import math
import tf_transformations
# from geometry_msgs.msg import PoseWithCovarianceStamped
from geometry_msgs.msg import PoseStamped

def get_quaternion_from_yaw(yaw_degrees):
  yaw_radians = math.radians(yaw_degrees)
  
  quaternion = tf_transformations.quaternion_from_euler(0, 0, yaw_radians)
  
  return quaternion

In [4]:
nav_points = {
  "init_point": (0, 0),
  "A0": (0.43, 1.0),
  "A1": (0.43, 0.0),
  "A2": (0.43, -0.5),
  "A3": (0.43, -1.2),
  "B0": (0.28, 1.0),
  "B1": (0.28, 0.0),
  "B2": (0.28, -0.5),
  "B3": (0.28, -1.2),
  "C0": (-0.45, 1.0),
  "C1": (-0.45, 0.0),
  "C2": (-0.45, -0.5),
  "C3": (-0.45, -1.2)
}

In [5]:
# 데모 시나리오 (PC에서 /goal_pose 토픽 발행)
sinario_demo_angle_degree = [0, 270, 90]
sinario_demo_pose = ["B1", "A0", "A3"]

# 맵을 한바뀌 돈 후에 원래 위치로 돌아오는 시나리오
sinario1_angle_degree = [0, 270, 90, 180, 270, 0, 135, 180]
sinario1_pose = ["B1", "A0", "A3", "C3", "C0", "A0", "B1", "init_point"]

In [6]:
# === [19.3-A] 속도 비율 조정 ===
LINE_SPEED_SCALE = 0.1  # 80% 속도 적용
ANGULAR_SPEED_SCALE = 0.5  # 80% 속도 적용

# controller_server 파라미터 변경
import subprocess
subprocess.run([
    "ros2", "param", "set", "/controller_server", "FollowPath.max_vel_x", str(0.25 * LINE_SPEED_SCALE)
])
subprocess.run([
    "ros2", "param", "set", "/controller_server", "FollowPath.max_vel_theta", str(1.0 * ANGULAR_SPEED_SCALE)
])

print(f"[INFO] Nav2 선속도 파라미터 비율 적용 완료: {LINE_SPEED_SCALE*100:.0f}%")
print(f"[INFO] Nav2 각속도 파라미터 비율 적용 완료: {ANGULAR_SPEED_SCALE*100:.0f}%")


Set parameter successful


KeyboardInterrupt: 

In [7]:
# 19.4 waypoint list 만들기
goal_pose_list = []

def double_tuple_to_posestamped_goal_pose(frame_id, pose_x, pose_y, yaw_degree):
  q = get_quaternion_from_yaw(yaw_degree)
  goal_pose = PoseStamped()
  goal_pose.header.frame_id = 'map'
  goal_pose.header.stamp = nav.get_clock().now().to_msg()
  goal_pose.pose.position.x = pose_x
  goal_pose.pose.position.y = pose_y
  goal_pose.pose.orientation.x = q[0]
  goal_pose.pose.orientation.y = q[1]
  goal_pose.pose.orientation.z = q[2]
  goal_pose.pose.orientation.w = q[3]
  return goal_pose

# sinario에 포함된 모든 nav points를 주행
for nav_point_pose, nav_point_angle in zip(sinario_demo_pose, sinario_demo_angle_degree):
  nav_point_pose_xy = nav_points[nav_point_pose]
  goal_pose_list.append(
    double_tuple_to_posestamped_goal_pose(
      'map', nav_point_pose_xy[0], nav_point_pose_xy[1], nav_point_angle
    )
  )

goal_pose_list

[geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1762515007, nanosec=188808448), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=0.28, y=0.0, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=0.0, y=0.0, z=0.0, w=1.0))),
 geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1762515007, nanosec=188927572), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=0.43, y=1.0, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=-0.0, y=0.0, z=0.7071067811865476, w=-0.7071067811865475))),
 geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1762515007, nanosec=189000158), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=0.43, y=-1.2, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=0.0, y=0.0, z=0.7071067811865475, w=0.7071067811865476)))]

In [6]:
# 19.4 waypoint list 만들기
goal_pose_list = []

def double_tuple_to_posestamped_goal_pose(frame_id, pose_x, pose_y, yaw_degree):
  q = get_quaternion_from_yaw(yaw_degree)
  goal_pose = PoseStamped()
  goal_pose.header.frame_id = 'map'
  goal_pose.header.stamp = nav.get_clock().now().to_msg()
  goal_pose.pose.position.x = pose_x
  goal_pose.pose.position.y = pose_y
  goal_pose.pose.orientation.x = q[0]
  goal_pose.pose.orientation.y = q[1]
  goal_pose.pose.orientation.z = q[2]
  goal_pose.pose.orientation.w = q[3]
  return goal_pose

# sinario에 포함된 모든 nav points를 주행
for nav_point_pose, nav_point_angle in zip(sinario1_pose, sinario1_angle_degree):
  nav_point_pose_xy = nav_points[nav_point_pose]
  goal_pose_list.append(
    double_tuple_to_posestamped_goal_pose(
      'map', nav_point_pose_xy[0], nav_point_pose_xy[1], nav_point_angle
    )
  )

goal_pose_list

[geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1762516540, nanosec=414550259), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=0.28, y=0.0, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=0.0, y=0.0, z=0.0, w=1.0))),
 geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1762516540, nanosec=414623127), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=0.43, y=1.0, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=-0.0, y=0.0, z=0.7071067811865476, w=-0.7071067811865475))),
 geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1762516540, nanosec=414667534), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=0.43, y=-1.2, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=0.0, y=0.0, z=0.7071067811865475, w=0.7071067811865476))),
 geometry_msgs.msg.PoseStamped(hea

In [ ]:
# 19.5 waypoint 주행 시작 후 피드백 확인
nav_start = nav.get_clock().now()
nav.followWaypoints(goal_pose_list)

: 